In [1]:
# =============================================================================
# FULL SCRIPT (Cairo-free): Manuscript-quality scaffold SVG panels
#
# Panel A: ChEMBL enriched antagonist scaffolds (diversity-constrained)
# Panel B: COCONUT high-mean-score scaffolds (diversity-constrained)
#
# Outputs (SVG vector):
#   diagnostics\fig_scaffolds_chembl.svg
#   diagnostics\fig_scaffolds_coconut.svg
#   diagnostics\fig_scaffolds_combined.svg
#
# Requirements (conda openms_env):
#   - rdkit
#   - pandas, numpy
# =============================================================================

import os, sys
print("exe:", sys.executable)
print("cwd:", os.getcwd())
import re
import numpy as np
import pandas as pd
import math
from rdkit import Chem, DataStructs
from rdkit.Chem import Draw
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.ML.Cluster import Butina
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit.Geometry import rdGeometry


# -----------------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------------
BASE = r"C:\Users\Besitzer\Desktop\M3_databases"

TRAIN_CSV = os.path.join(BASE, "ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv")
COCO_CSV  = os.path.join(BASE, "coconut_screen_out", "coconut_screen_ranked.csv")

OUTDIR = os.path.join(BASE, "diagnostics")
os.makedirs(OUTDIR, exist_ok=True)

OUT_SVG_A  = os.path.join(OUTDIR, "fig_scaffolds_chembl.svg")
OUT_SVG_B  = os.path.join(OUTDIR, "fig_scaffolds_coconut.svg")
OUT_SVG_AB = os.path.join(OUTDIR, "fig_scaffolds_combined.svg")


# -----------------------------------------------------------------------------
# COLUMN MAP (edit if your columns differ)
# -----------------------------------------------------------------------------
TRAIN_LABEL_COL  = "consensus_label"
TRAIN_SMILES_PREF = "canonical_smiles"   # falls back to smiles/SMILES
COCO_SMILES_PREF  = "smiles"             # falls back to canonical_smiles/smiles/SMILES
COCO_SCORE_COL    = "p_antagonist"

POS_LABELS = {"active", "active_single"}   # operational "positive" class


# -----------------------------------------------------------------------------
# SELECTION SETTINGS
# -----------------------------------------------------------------------------
N_SHOW = 10

# Evidence thresholds
CHEMBL_MIN_N = 5
CHEMBL_MIN_POS = 3
COCO_MIN_N = 10

# Diversity clustering
FP_RADIUS = 2
FP_BITS   = 2048
BUTINA_CUTOFF = 0.65  # 0.6–0.7 typical; higher => fewer clusters => more diversity

# Rendering layout
N_COLS = 5
CELL_W, CELL_H = 420, 340  # per structure tile in SVG


# =============================================================================
# Helpers
# =============================================================================

def get_smiles_col(df: pd.DataFrame, preferred: str) -> str:
    if preferred in df.columns:
        return preferred
    for c in ["canonical_smiles", "smiles", "SMILES"]:
        if c in df.columns:
            return c
    raise ValueError("No SMILES column found. Expected one of canonical_smiles/smiles/SMILES.")

def smiles_to_mol(smi: str):
    if not isinstance(smi, str) or not smi.strip():
        return None
    try:
        return Chem.MolFromSmiles(smi)
    except Exception:
        return None

def murcko_scaffold_smiles(mol):
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None or scaf.GetNumAtoms() == 0:
            return None
        return Chem.MolToSmiles(scaf, isomericSmiles=False)
    except Exception:
        return None

def scaffold_to_fp(scaf_smi: str):
    m = Chem.MolFromSmiles(scaf_smi) if isinstance(scaf_smi, str) else None
    if m is None:
        return None
    return rdMolDescriptors.GetMorganFingerprintAsBitVect(m, radius=FP_RADIUS, nBits=FP_BITS)

def butina_cluster(fps, cutoff=0.65):
    n = len(fps)
    if n == 0:
        return []
    dists = []
    for i in range(1, n):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1.0 - x for x in sims])
    # Butina expects distances; distThresh is distance cutoff = 1 - similarity_cutoff
    clusters = Butina.ClusterData(dists, nPts=n, distThresh=1.0 - cutoff, isDistData=True)
    return [list(c) for c in clusters]

def save_text(path: str, text: str):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def escape_xml(s: str) -> str:
    return (s.replace("&","&amp;")
             .replace("<","&lt;")
             .replace(">","&gt;")
             .replace('"',"&quot;")
             .replace("'","&apos;"))

def svg_get_size(svg_text):
    """
    Returns (W, H) as floats in "pixel-like" units.
    Tries width/height attributes first, then falls back to viewBox.
    """
    if not svg_text:
        return (None, None)

    # 1) Try width/height attributes (allow px, pt, mm, cm, in, or no unit)
    m_w = re.search(r"\bwidth\s*=\s*['\"]\s*([0-9]*\.?[0-9]+)\s*([a-zA-Z%]*)\s*['\"]", svg_text)
    m_h = re.search(r"\bheight\s*=\s*['\"]\s*([0-9]*\.?[0-9]+)\s*([a-zA-Z%]*)\s*['\"]", svg_text)

    def to_px(val, unit):
        val = float(val)
        unit = (unit or "").lower()
        # We only need consistent relative units to combine; assume:
        # px: 1, pt: 1.333..., in: 96, cm: 37.795..., mm: 3.7795
        if unit == "" or unit == "px":
            return val
        if unit == "pt":
            return val * (96.0 / 72.0)
        if unit == "in":
            return val * 96.0
        if unit == "cm":
            return val * (96.0 / 2.54)
        if unit == "mm":
            return val * (96.0 / 25.4)
        # % or unknown -> can't convert reliably
        return None

    if m_w and m_h:
        W = to_px(m_w.group(1), m_w.group(2))
        H = to_px(m_h.group(1), m_h.group(2))
        if W is not None and H is not None:
            return (W, H)

    # 2) Fallback: viewBox="minx miny width height"
    m_vb = re.search(r"\bviewBox\s*=\s*['\"]\s*([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s+([-\d\.]+)\s*['\"]", svg_text)
    if m_vb:
        W = float(m_vb.group(3))
        H = float(m_vb.group(4))
        return (W, H)

    return (None, None)

def svg_strip_outer(svg_text: str) -> str:
    start = svg_text.find(">")
    end = svg_text.rfind("</svg>")
    if start == -1 or end == -1:
        return svg_text
    return svg_text[start+1:end].strip()

def wrap_with_title(svg_text: str, title: str, subtitle: str = None,
                    pad: int = 24, font_title: int = 26, font_sub: int = 18):
    W, H = svg_get_size(svg_text)
    if W is None:
        return svg_text

    title_block = pad + font_title*1.6 + (font_sub*1.4 if subtitle else 0) + pad
    inner = svg_strip_outer(svg_text)

    y1 = pad + font_title
    y2 = y1 + font_title*1.2

    out = []
    out.append(f'<svg xmlns="http://www.w3.org/2000/svg" width="{W}px" height="{H+title_block}px" viewBox="0 0 {W} {H+title_block}">')
    out.append(f'<text x="{pad}" y="{y1}" font-family="Arial, Helvetica, sans-serif" font-size="{font_title}" fill="black">{escape_xml(title)}</text>')
    if subtitle:
        out.append(f'<text x="{pad}" y="{y2}" font-family="Arial, Helvetica, sans-serif" font-size="{font_sub}" fill="black">{escape_xml(subtitle)}</text>')
    out.append(f'<g transform="translate(0,{title_block})">{inner}</g>')
    out.append('</svg>')
    return "\n".join(out)

def combine_svgs_vert(svg_top, svg_bottom, gap=50):
    W1, H1 = svg_get_size(svg_top)
    W2, H2 = svg_get_size(svg_bottom)
    if W1 is None or W2 is None or H1 is None or H2 is None:
        raise ValueError("Could not parse SVG width/height (or viewBox) for combining.")

    W = max(W1, W2)
    H = H1 + gap + H2

    def inner(svg):
        # remove outer <svg ...> ... </svg>, keep inner content
        # Works for typical RDKit SVG.
        svg = re.sub(r"^\s*<\?xml[^>]*>\s*", "", svg, flags=re.I)
        svg = re.sub(r"^\s*<!DOCTYPE[^>]*>\s*", "", svg, flags=re.I)
        svg = re.sub(r"^\s*<svg[^>]*>", "", svg, flags=re.I)
        svg = re.sub(r"</svg>\s*$", "", svg, flags=re.I)
        return svg.strip()

    top_inner = inner(svg_top)
    bot_inner = inner(svg_bottom)

    # build combined svg with explicit width/height + viewBox
    out = f"""<svg xmlns="http://www.w3.org/2000/svg" width="{W}px" height="{H}px" viewBox="0 0 {W} {H}">
<g transform="translate(0,0)">
{top_inner}
</g>
<g transform="translate(0,{H1 + gap})">
{bot_inner}
</g>
</svg>"""
    return out

def render_svg_grid(mols, legends,
                    n_cols=4, cell_w=240, cell_h=180,
                    legend_font=14):

    mols2, leg2 = [], []
    for m, l in zip(mols, legends):
        if m is None:
            continue
        if isinstance(m, str):
            m = Chem.MolFromSmiles(m)
        if m is None:
            continue
        mols2.append(m)
        leg2.append("" if l is None else str(l))

    n = len(mols2)
    if n == 0:
        return ""

    n_rows = (n + n_cols - 1) // n_cols
    W = int(n_cols * cell_w)
    H = int(n_rows * cell_h)

    drawer = rdMolDraw2D.MolDraw2DSVG(W, H, int(cell_w), int(cell_h))
    opts = drawer.drawOptions()
    opts.legendFontSize = int(legend_font)
    opts.padding = 0.1

    drawer.DrawMolecules(
        mols2,
        legends=leg2,
        highlightAtoms=[[]]*n,
        highlightBonds=[[]]*n,
        confIds=[-1]*n
    )
    drawer.FinishDrawing()
    return drawer.GetDrawingText()

from xml.sax.saxutils import escape

def wrap_with_title(svg, title, subtitle,
                    padding=20, title_font=18, subtitle_font=14, line_gap=6,
                    font_family="Arial"):

    if not svg:
        return svg

    title = escape("" if title is None else str(title))
    subtitle = escape("" if subtitle is None else str(subtitle))

    # width/height auslesen
    m_w = re.search(r"width=['\"]([\d\.]+)(px)?['\"]", svg)
    m_h = re.search(r"height=['\"]([\d\.]+)(px)?['\"]", svg)
    if not (m_w and m_h):
        # fallback: Text einfach nach <svg ...> rein
        insert = (
            f"\n<text x='{padding}' y='{padding+title_font}' font-size='{title_font}' font-family='{font_family}'>{title}</text>"
            f"\n<text x='{padding}' y='{padding+title_font+subtitle_font+line_gap}' font-size='{subtitle_font}' font-family='{font_family}'>{subtitle}</text>\n"
        )
        return svg.replace(">", ">" + insert, 1)

    W = float(m_w.group(1))
    H = float(m_h.group(1))

    header_h = padding + title_font + subtitle_font + line_gap + padding
    new_H = H + header_h

    # height ersetzen
    svg2 = re.sub(r"height=['\"][\d\.]+(px)?['\"]", f"height='{new_H}px'", svg, count=1)

    header = (
        f"\n<text x='{padding}' y='{padding+title_font}' font-size='{title_font}' font-family='{font_family}'>{title}</text>"
        f"\n<text x='{padding}' y='{padding+title_font+subtitle_font+line_gap}' font-size='{subtitle_font}' font-family='{font_family}'>{subtitle}</text>\n"
        f"<g transform='translate(0,{header_h})'>\n"
    )

    svg2 = svg2.replace(">", ">" + header, 1)
    svg2 = svg2.replace("</svg>", "\n</g>\n</svg>", 1)
    return svg2

# =============================================================================
# 1) Load ChEMBL and compute scaffolds
# =============================================================================

print("[LOAD TRAIN]", TRAIN_CSV)
train = pd.read_csv(TRAIN_CSV)
train_smi_col = get_smiles_col(train, TRAIN_SMILES_PREF)

if TRAIN_LABEL_COL not in train.columns:
    raise ValueError(f"Missing label column '{TRAIN_LABEL_COL}' in training CSV.")

train["is_pos"] = train[TRAIN_LABEL_COL].astype(str).isin(POS_LABELS).astype(int)

if "scaffold" not in train.columns:
    print("[TRAIN] 'scaffold' missing -> computing Murcko scaffolds...")
    mols = [smiles_to_mol(s) for s in train[train_smi_col].astype(str)]
    train["scaffold"] = [murcko_scaffold_smiles(m) for m in mols]
else:
    train["scaffold"] = train["scaffold"].astype(str)

train = train.dropna(subset=["scaffold"]).copy()
train = train[train["scaffold"].str.len() > 0].copy()

p_global = float(train["is_pos"].mean())
print(f"[TRAIN] n={len(train)} | p_global={p_global:.3f} | unique scaffolds={train['scaffold'].nunique()}")

# Per-scaffold enrichment stats
g = train.groupby("scaffold").agg(
    n=("is_pos", "size"),
    n_pos=("is_pos", "sum")
).reset_index()

g["p_scaf"] = g["n_pos"] / g["n"]
g["enrich"] = g["p_scaf"] / p_global
g["log_enrich"] = np.log(g["enrich"].replace(0, np.nan))

# Evidence filter
g_f = g[(g["n"] >= CHEMBL_MIN_N) & (g["n_pos"] >= CHEMBL_MIN_POS)].copy()
g_f = g_f.sort_values(["log_enrich", "n"], ascending=[False, False]).reset_index(drop=True)

if len(g_f) == 0:
    raise RuntimeError("No ChEMBL scaffolds passed evidence filters. Lower CHEMBL_MIN_N / CHEMBL_MIN_POS.")

# Add fingerprints for clustering
g_f["fp"] = g_f["scaffold"].apply(scaffold_to_fp)
g_f = g_f.dropna(subset=["fp"]).copy()

# Diversity selection: cluster scaffolds, pick best per cluster
fps = list(g_f["fp"].values)
clusters = butina_cluster(fps, cutoff=BUTINA_CUTOFF)

selected = []
for cl in clusters:
    sub = g_f.iloc[cl].copy().sort_values(["log_enrich", "n"], ascending=[False, False])
    selected.append(sub.iloc[0])

chembl_sel = pd.DataFrame(selected).sort_values(["log_enrich", "n"], ascending=[False, False]).head(N_SHOW).copy()
print(f"[CHEMBL] selected {len(chembl_sel)} scaffolds (diversity-constrained)")

chembl_mols = [Chem.MolFromSmiles(s) for s in chembl_sel["scaffold"]]
chembl_legends = [
    f"n={int(r.n)} | pos={int(r.n_pos)} ({100*float(r.p_scaf):.1f}%)\n"
    f"enrich×={float(r.enrich):.2f}"
    for r in chembl_sel.itertuples(index=False)
]

# =============================================================================
# 2) Load COCONUT ranked and compute scaffolds
# =============================================================================

print("[LOAD COCONUT]", COCO_CSV)
coco = pd.read_csv(COCO_CSV)
coco_smi_col = get_smiles_col(coco, COCO_SMILES_PREF)

if COCO_SCORE_COL not in coco.columns:
    raise ValueError(f"Missing score column '{COCO_SCORE_COL}' in COCONUT ranked CSV.")

print("[COCONUT] computing Murcko scaffolds...")
mols = [smiles_to_mol(s) for s in coco[coco_smi_col].astype(str)]
coco["scaffold"] = [murcko_scaffold_smiles(m) for m in mols]

coco = coco.dropna(subset=["scaffold", COCO_SCORE_COL]).copy()
coco = coco[coco["scaffold"].str.len() > 0].copy()
print(f"[COCONUT] n_ok={len(coco)} | unique scaffolds={coco['scaffold'].nunique()}")

# Per-scaffold mean score
h = coco.groupby("scaffold").agg(
    n=(COCO_SCORE_COL, "size"),
    p_mean=(COCO_SCORE_COL, "mean"),
    p_median=(COCO_SCORE_COL, "median")
).reset_index()

h_f = h[h["n"] >= COCO_MIN_N].copy()
h_f = h_f.sort_values(["p_mean", "n"], ascending=[False, False]).reset_index(drop=True)

if len(h_f) == 0:
    raise RuntimeError("No COCONUT scaffolds passed COCO_MIN_N. Lower COCO_MIN_N.")

h_f["fp"] = h_f["scaffold"].apply(scaffold_to_fp)
h_f = h_f.dropna(subset=["fp"]).copy()

fps = list(h_f["fp"].values)
clusters = butina_cluster(fps, cutoff=BUTINA_CUTOFF)

selected = []
for cl in clusters:
    sub = h_f.iloc[cl].copy().sort_values(["p_mean", "n"], ascending=[False, False])
    selected.append(sub.iloc[0])

coco_sel = pd.DataFrame(selected).sort_values(["p_mean", "n"], ascending=[False, False]).head(N_SHOW).copy()
print(f"[COCONUT] selected {len(coco_sel)} scaffolds (diversity-constrained)")

coco_mols = [Chem.MolFromSmiles(s) for s in coco_sel["scaffold"]]
coco_legends = [
    f"n={int(r.n)} | mean p={float(r.p_mean):.3f}\nmedian p={float(r.p_median):.3f}"
    for r in coco_sel.itertuples(index=False)
]

# =============================================================================
# 3) Render SVG panels (NO Cairo)
# =============================================================================

title_A = "Panel A — ChEMBL: enriched antagonist scaffolds (diversity-constrained)"
subtitle_A = f"Rank: log-enrichment vs global pos-rate (p_global={p_global:.3f}); filters: n≥{CHEMBL_MIN_N}, pos≥{CHEMBL_MIN_POS}; Butina cutoff={BUTINA_CUTOFF}"

svg_A = render_svg_grid(
    chembl_mols, chembl_legends,n_cols=N_COLS, cell_w=CELL_W, cell_h=CELL_H
)

title_B = "Panel B — COCONUT: high-score scaffolds (diversity-constrained)"
subtitle_B = f"Rank: mean p_antagonist per scaffold; filter: n≥{COCO_MIN_N}; Butina cutoff={BUTINA_CUTOFF}"

svg_B = render_svg_grid(coco_mols, coco_legends, n_cols=N_COLS, cell_w=CELL_W, cell_h=CELL_H)
svg_B = wrap_with_title(svg_B, title_B, subtitle_B)

save_text(OUT_SVG_A, svg_A)
save_text(OUT_SVG_B, svg_B)

svg_AB = combine_svgs_vert(svg_A, svg_B, gap=50)
save_text(OUT_SVG_AB, svg_AB)

print("Saved SVGs:")
print(" -", OUT_SVG_A)
print(" -", OUT_SVG_B)
print(" -", OUT_SVG_AB)


exe: c:\Users\Besitzer\anaconda3\envs\openms_env\python.exe
cwd: c:\Users\Besitzer\Desktop\M3_databases
[LOAD TRAIN] C:\Users\Besitzer\Desktop\M3_databases\ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv
[TRAIN] 'scaffold' missing -> computing Murcko scaffolds...
[TRAIN] n=2268 | p_global=0.788 | unique scaffolds=1022
[CHEMBL] selected 10 scaffolds (diversity-constrained)
[LOAD COCONUT] C:\Users\Besitzer\Desktop\M3_databases\coconut_screen_out\coconut_screen_ranked.csv


[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerator
[23:19:53] DEPRECATION WARNING: please use MorganGenerat

[COCONUT] computing Murcko scaffolds...


[23:20:03] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 10 11 15 16 17 19 20 21
[23:20:03] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19


[COCONUT] n_ok=385984 | unique scaffolds=109061


[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerator
[23:21:33] DEPRECATION WARNING: please use MorganGenerat

[COCONUT] selected 10 scaffolds (diversity-constrained)
Saved SVGs:
 - C:\Users\Besitzer\Desktop\M3_databases\diagnostics\fig_scaffolds_chembl.svg
 - C:\Users\Besitzer\Desktop\M3_databases\diagnostics\fig_scaffolds_coconut.svg
 - C:\Users\Besitzer\Desktop\M3_databases\diagnostics\fig_scaffolds_combined.svg
